In [1]:
import pandas as pd
import glob
import os

In [2]:
def read_csv_auto(file):
    try:
        return pd.read_csv(file, encoding='utf-8')
    except:
        return pd.read_csv(file, encoding='cp949')

def load_and_concat(folder):
    csv_files = glob.glob(os.path.join(folder, '*.csv'))
    df_list = []

    for file in csv_files:
        df = read_csv_auto(file)

        if 'bike' in folder.lower():
            df = df[['대여일시', '대여 대여소명', '대여 대여소번호']]
        else:
            raise ValueError(f"Unknown folder type: {folder}")

        df_list.append(df)

    return pd.concat(df_list, ignore_index=True)

In [24]:
def read_subway_adaptive(file):
    columns = ['사용일자', '노선명', '역명', '승차총승객수', '하차총승객수', '등록일자', '기타']
    filename = os.path.basename(file)

    for encoding in ['utf-8-sig', 'cp949']:
        try:
            if '202402' in filename:
                # 예외 처리용: 열 밀림 보정
                df = pd.read_csv(file, encoding=encoding, quotechar='"', header=None, skiprows=1, names=columns)
            else:
                df = pd.read_csv(file, encoding=encoding, quotechar='"', header=0, names=columns)
            return df[['사용일자', '역명', '승차총승객수']]
        except:
            continue
    raise ValueError(f"파일 구조 이상: {file}")

In [3]:
bike = load_and_concat("bike")

In [23]:
subway_files = sorted([f for f in glob.glob(os.path.join("subway", "*.csv")) if "SUBWAY" in f])

In [25]:
subway_df_list = [read_subway_adaptive(file) for file in subway_files]
subway_final_fixed = pd.concat(subway_df_list, ignore_index=True)

In [9]:
# '\\N' 제거
bike = bike[bike["대여 대여소번호"] != "\\N"]

# 정수로 변환
bike["대여 대여소번호"] = bike["대여 대여소번호"].astype(int)

# 5자리 문자열로 포맷
bike["대여 대여소번호"] = bike["대여 대여소번호"].apply(lambda x: f"{x:05d}")

In [7]:
def uni_df(dataframe):
    for col in dataframe:
        print(col,dataframe[col].unique())
        print(col,dataframe[col].nunique())

In [13]:
bike['대여일시'] = pd.to_datetime(bike['대여일시']).dt.date

In [16]:
# 결과 확인
print(bike.head())
print("Folder1 shape:", bike.shape)
uni_df(bike)

         대여일시          대여 대여소명 대여 대여소번호
0  2024-01-01          동서울농협 앞    04804
1  2024-01-01           신대방삼거리    04169
2  2024-01-01  군자역 7번출구 베스트샵 앞    00540
3  2024-01-01        용문사 버스정류장    01139
4  2024-01-01        동묘앞역 6번출구    03416
Folder1 shape: (44360554, 3)
대여일시 [datetime.date(2024, 1, 1) datetime.date(2024, 1, 2)
 datetime.date(2024, 1, 3) datetime.date(2024, 1, 4)
 datetime.date(2024, 1, 5) datetime.date(2024, 1, 6)
 datetime.date(2024, 1, 7) datetime.date(2024, 1, 8)
 datetime.date(2024, 1, 9) datetime.date(2024, 1, 10)
 datetime.date(2024, 1, 11) datetime.date(2024, 1, 12)
 datetime.date(2024, 1, 13) datetime.date(2024, 1, 14)
 datetime.date(2024, 1, 15) datetime.date(2024, 1, 16)
 datetime.date(2024, 1, 17) datetime.date(2024, 1, 18)
 datetime.date(2024, 1, 19) datetime.date(2024, 1, 20)
 datetime.date(2024, 1, 21) datetime.date(2024, 1, 22)
 datetime.date(2024, 1, 23) datetime.date(2024, 1, 24)
 datetime.date(2024, 1, 25) datetime.date(2024, 1, 26)
 datetime.date(2024,

In [ ]:
print(subway_final_fixed.head())
print("Folder2 shape:", subway_final_fixed.shape)

In [12]:
subway_final_fixed[subway_final_fixed['사용일자'] == 20240104].head()

NameError: name 'subway_final_fixed' is not defined

In [15]:
bike.to_csv('bike_merged.csv', index=False, encoding='utf-8-sig')

In [ ]:
subway_final_fixed.to_csv('subway_merged.csv', index=False, encoding='utf-8-sig')